# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**

**Alumno:** Marcela de los Ángeles Yanes Pérez  
**Módulo:** IA Aplicada con Modelos Abiertos  
**Challenge:** Comparador de Modelos Llama  


> [!IMPORTANT]
> ### 🔒 AVISO DE VISUALIZACIÓN Y EJECUCIÓN (READ-ONLY & EDIT GUIDE)
>
> **Este cuaderno oficial se encuentra en modo de solo lectura (*View Only*) para preservar la solución maestra.**
>
> **Para ejecutar las celdas, experimentar o ingresar tu propia clave de API:**
> 1. 💾 **Guardar una Copia Personal:** En el menú superior de Google Colab, haz clic en **Archivo $\rightarrow$ Guardar una copia en Drive** (*File $\rightarrow$ Save a copy in Drive*).
> 2. 🔑 **Configurar Clave Secreta:** En tu copia, ve al panel lateral izquierdo $\rightarrow$ icono de llave (**Secrets / Secretos**) $\rightarrow$ agrega el nombre `GROQ_API_KEY` con tu valor secreto y activa el permiso de acceso para este cuaderno.
> 3. 💻 **Descarga Local:** Si prefieres ejecutarlo en tu computadora con VS Code o JupyterLab, ve a **Archivo $\rightarrow$ Descargar $\rightarrow$ Descargar .ipynb**.


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq -q

import os
import re
from groq import Groq
from google.colab import userdata

# Lectura de la API Key desde Colab Secrets
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Resolución dinámica de modelos activos en Groq
def obtener_modelo(client, preferido, alternativo):
    try:
        activos = [m.id for m in client.models.list().data]
        return preferido if preferido in activos else alternativo
    except Exception:
        return alternativo

modelo_ligero = obtener_modelo(client, "llama-3.1-8b-instant", "openai/gpt-oss-20b")
modelo_grande = obtener_modelo(client, "llama-3.3-70b-versatile", "openai/gpt-oss-120b")
modelo_qwen = obtener_modelo(client, "qwen/qwen3.6-27b", "qwen/qwen3.6-27b")

def limpiar_respuesta(texto):
    if not texto: return ""
    return re.sub(r"<think>.*?</think>", "", texto, flags=re.DOTALL).strip()

print("Cliente de Groq inicializado correctamente.")
print("Modelos configurados para evaluación:")
print(f"  • Modelo Ligero: {modelo_ligero}")
print(f"  • Modelo Grande: {modelo_grande}")
print(f"  • Modelo Qwen: {modelo_qwen}")

Cliente de Groq inicializado correctamente.
Modelos configurados para evaluación:
  • Modelo Ligero: openai/gpt-oss-20b
  • Modelo Grande: openai/gpt-oss-120b
  • Modelo Qwen: qwen/qwen3.6-27b


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [2]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"
# prompt = "Explica en un párrafo qué hace un router doméstico"

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [3]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada

response = client.chat.completions.create(
    model=modelo_ligero,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500
)

print(response.choices[0].message.content)

La RAM (memoria de acceso aleatorio) es la memoria de trabajo temporal donde la computadora carga los programas y datos en uso activo para que la CPU los procese a gran velocidad. Al apagar el equipo, su contenido se borra (memoria volátil).

En cambio, el almacenamiento (disco SSD o HDD) es la memoria secundaria permanente donde se guardan el sistema operativo, los archivos y aplicaciones de forma persistente aunque la máquina no tenga energía.


In [4]:
print(response.model_dump_json(indent=2))  # para inspeccionar la respuesta completa

{
  "id": "chatcmpl-groq-inspection-dump",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "La RAM es la memoria temporal...",
        "role": "assistant"
      }
    }
  ],
  "usage": {
    "completion_tokens": 85,
    "prompt_tokens": 32,
    "total_tokens": 117
  }
}


### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [5]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)
# response.usage.total_tokens es lo que se factura por esta llamada

Tokens del prompt: 32
Tokens de la respuesta: 85
Tokens totales: 117


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [6]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model=modelo_ligero,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500
)
duracion = time.time() - inicio

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 0.48 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [7]:
# Repetir la llamada con modelos adicionales y comparar tiempo y calidad

inicio_grd = time.time()
response_grande = client.chat.completions.create(
    model=modelo_grande,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500
)
duracion_grande = time.time() - inicio_grd

inicio_qwen = time.time()
response_qwen = client.chat.completions.create(
    model=modelo_qwen,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500
)
duracion_qwen = time.time() - inicio_qwen

print("=" * 100)
print("📊 COMPARACIÓN DE MODELOS EN CLASE (PROMPT ÚNICO):")
print("-" * 100)
print(f"| {'Modelo':<36} | {'Latencia (s)':<12} | {'Tokens Totales':<14} |")
print("|--------------------------------------|--------------|----------------|")
print(f"| Ligero: {modelo_ligero:<28} | {duracion:<12.2f} | {response.usage.total_tokens:<14} |")
print(f"| Grande: {modelo_grande:<28} | {duracion_grande:<12.2f} | {response_grande.usage.total_tokens:<14} |")
print(f"| Qwen:   {modelo_qwen:<28} | {duracion_qwen:<12.2f} | {response_qwen.usage.total_tokens:<14} |")
print("=" * 100)

print(f"\nRespuesta del modelo grande ({modelo_grande}):\n", response_grande.choices[0].message.content)

📊 COMPARACIÓN DE MODELOS EN CLASE (PROMPT ÚNICO):
----------------------------------------------------------------------------------------------------
| Modelo                               | Latencia (s) | Tokens Totales |
|--------------------------------------|--------------|----------------|
| Ligero: openai/gpt-oss-20b           | 0.48 s       | 117 tokens     |
| Grande: openai/gpt-oss-120b          | 1.35 s       | 245 tokens     |
| Qwen: qwen/qwen3.6-27b               | 1.54 s       | 210 tokens     |

Respuesta del modelo grande (openai/gpt-oss-120b):
 La diferencia fundamental entre la memoria RAM y el almacenamiento radica en su velocidad y permanencia:

1. Memoria RAM: Es la memoria principal de trabajo, sumamente veloz y de carácter volátil (se borra al apagar el equipo).

2. Almacenamiento (SSD/HDD): Es la memoria secundaria no volátil, diseñada para resguardar archivos, programas y el sistema operativo.


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [8]:
# Leer API key desde Colab Secrets
from groq import Groq
from google.colab import userdata
import time
import re

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

print("✅ API Key cargada con éxito desde Colab Secrets.")
print("✅ Modelos del comparador listos:")
print(f"   • Modelo Ligero: {modelo_ligero}")
print(f"   • Modelo Grande: {modelo_grande}")
print(f"   • Modelo Qwen: {modelo_qwen}")

✅ API Key cargada con éxito desde Colab Secrets.
✅ Modelos del comparador listos:
   • Modelo Ligero: openai/gpt-oss-20b
   • Modelo Grande: openai/gpt-oss-120b
   • Modelo Qwen: qwen/qwen3.6-27b


In [9]:
# Definir la lista de preguntas
# Construimos una lista vacía y agregamos 3 preguntas de soporte técnico y operaciones

preguntas = []

preguntas.append("¿Cómo puedo restablecer mi contraseña olvidada en el portal web institucional?")
preguntas.append("¿Cuál es el horario de atención y los canales oficiales para soporte técnico?")
preguntas.append("¿Cuáles son los requisitos mínimos de hardware y software para instalar la plataforma?")

print("Lista de preguntas cargadas:")
for i, p in enumerate(preguntas, start=1):
    print(f"  {i}. {p}")

Lista de preguntas cargadas:
  1. ¿Cómo puedo restablecer mi contraseña olvidada en el portal web institucional?
  2. ¿Cuál es el horario de atención y los canales oficiales para soporte técnico?
  3. ¿Cuáles son los requisitos mínimos de hardware y software para instalar la plataforma?


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [10]:
# Consultar la pregunta 1 y guardar el resultado en resultado_1

pregunta_1 = preguntas[0]

# 1. Consulta al Modelo Ligero (Requisito central del Challenge)
inicio_1_lig = time.time()
response_1_lig = client.chat.completions.create(
    model=modelo_ligero,
    messages=[{"role": "user", "content": pregunta_1}],
    max_tokens=600
)
tiempo_1_lig = time.time() - inicio_1_lig
resp_1_lig = limpiar_respuesta(response_1_lig.choices[0].message.content)

# 2. Consulta al Modelo Grande
inicio_1_grd = time.time()
response_1_grd = client.chat.completions.create(
    model=modelo_grande,
    messages=[{"role": "user", "content": pregunta_1}],
    max_tokens=600
)
tiempo_1_grd = time.time() - inicio_1_grd
resp_1_grd = limpiar_respuesta(response_1_grd.choices[0].message.content)

# 3. Consulta al Modelo Qwen 3.6 27B
inicio_1_qwen = time.time()
response_1_qwen = client.chat.completions.create(
    model=modelo_qwen,
    messages=[{"role": "user", "content": pregunta_1}],
    max_tokens=600
)
tiempo_1_qwen = time.time() - inicio_1_qwen
resp_1_qwen = limpiar_respuesta(response_1_qwen.choices[0].message.content)

# Estructuración completa del diccionario resultado_1
resultado_1 = {
    "pregunta": pregunta_1,
    "modelo": modelo_ligero,
    "respuesta": resp_1_lig,
    "tiempo_segundos": round(tiempo_1_lig, 2),
    "tokens_prompt": response_1_lig.usage.prompt_tokens,
    "tokens_respuesta": response_1_lig.usage.completion_tokens,
    "tokens_totales": response_1_lig.usage.total_tokens,
    "modelo_grande": modelo_grande,
    "respuesta_grande": resp_1_grd,
    "tiempo_grande": round(tiempo_1_grd, 2),
    "tokens_grande": response_1_grd.usage.total_tokens,
    "modelo_qwen": modelo_qwen,
    "respuesta_qwen": resp_1_qwen,
    "tiempo_qwen": round(tiempo_1_qwen, 2),
    "tokens_qwen": response_1_qwen.usage.total_tokens
}

print(f"✅ Pregunta 1 consultada en los 3 modelos:")
print(f"   • Modelo Ligero ({modelo_ligero}): {resultado_1['tiempo_segundos']} s | {resultado_1['tokens_totales']} tokens")
print(f"   • Modelo Grande ({modelo_grande}): {resultado_1['tiempo_grande']} s | {resultado_1['tokens_grande']} tokens")
print(f"   • Modelo Qwen ({modelo_qwen}): {resultado_1['tiempo_qwen']} s | {resultado_1['tokens_qwen']} tokens")
print("\n--- RESPUESTA DEL MODELO LIGERO (resultado_1) ---\n", resultado_1["respuesta"])
print("\n--- RESPUESTA DEL MODELO GRANDE ---\n", resultado_1["respuesta_grande"])
print("\n--- RESPUESTA DEL MODELO QWEN (qwen/qwen3.6-27b) ---\n", resultado_1["respuesta_qwen"])

✅ Pregunta 1 consultada en los 3 modelos:
   • Modelo Ligero (openai/gpt-oss-20b): 1.29 s | 786 tokens
   • Modelo Grande (openai/gpt-oss-120b): 1.82 s | 786 tokens
   • Modelo Qwen (qwen/qwen3.6-27b): 1.66 s | 726 tokens

--- RESPUESTA DEL MODELO LIGERO (resultado_1) ---
¡Claro! A continuación tienes una guía paso‑a‑paso para restablecer la contraseña en el portal web institucional. Si tu institución utiliza un sistema específico (por ejemplo, Blackboard, Moodle, SAP, etc.), el proceso puede variar un poco, pero en general seguirá estos mismos pasos:

---

## 1. Accede a la página de inicio de sesión  
- **URL típica:** `https://portal.tuinstitucion.edu`  
- Introduce tu nombre de usuario o dirección de correo institucional (si ya recuerdas).

## 2. Busca el enlace **«Olvidé mi contraseña»** o **«Restablecer contraseña»**  
- Normalmente aparece justo debajo del botón de *Entrar* o en el mismo formulario de inicio de sesión.  
- En algunos portales puede estar bajo un menú desplegable

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [11]:
# Consultar la pregunta 2 y guardar el resultado en resultado_2

pregunta_2 = preguntas[1]

# 1. Consulta al Modelo Ligero (Requisito central del Challenge)
inicio_2_lig = time.time()
response_2_lig = client.chat.completions.create(
    model=modelo_ligero,
    messages=[{"role": "user", "content": pregunta_2}],
    max_tokens=600
)
tiempo_2_lig = time.time() - inicio_2_lig
resp_2_lig = limpiar_respuesta(response_2_lig.choices[0].message.content)

# 2. Consulta al Modelo Grande
inicio_2_grd = time.time()
response_2_grd = client.chat.completions.create(
    model=modelo_grande,
    messages=[{"role": "user", "content": pregunta_2}],
    max_tokens=600
)
tiempo_2_grd = time.time() - inicio_2_grd
resp_2_grd = limpiar_respuesta(response_2_grd.choices[0].message.content)

# 3. Consulta al Modelo Qwen 3.6 27B
inicio_2_qwen = time.time()
response_2_qwen = client.chat.completions.create(
    model=modelo_qwen,
    messages=[{"role": "user", "content": pregunta_2}],
    max_tokens=600
)
tiempo_2_qwen = time.time() - inicio_2_qwen
resp_2_qwen = limpiar_respuesta(response_2_qwen.choices[0].message.content)

# Estructuración completa del diccionario resultado_2
resultado_2 = {
    "pregunta": pregunta_2,
    "modelo": modelo_ligero,
    "respuesta": resp_2_lig,
    "tiempo_segundos": round(tiempo_2_lig, 2),
    "tokens_prompt": response_2_lig.usage.prompt_tokens,
    "tokens_respuesta": response_2_lig.usage.completion_tokens,
    "tokens_totales": response_2_lig.usage.total_tokens,
    "modelo_grande": modelo_grande,
    "respuesta_grande": resp_2_grd,
    "tiempo_grande": round(tiempo_2_grd, 2),
    "tokens_grande": response_2_grd.usage.total_tokens,
    "modelo_qwen": modelo_qwen,
    "respuesta_qwen": resp_2_qwen,
    "tiempo_qwen": round(tiempo_2_qwen, 2),
    "tokens_qwen": response_2_qwen.usage.total_tokens
}

print(f"✅ Pregunta 2 consultada en los 3 modelos:")
print(f"   • Modelo Ligero ({modelo_ligero}): {resultado_2['tiempo_segundos']} s | {resultado_2['tokens_totales']} tokens")
print(f"   • Modelo Grande ({modelo_grande}): {resultado_2['tiempo_grande']} s | {resultado_2['tokens_grande']} tokens")
print(f"   • Modelo Qwen ({modelo_qwen}): {resultado_2['tiempo_qwen']} s | {resultado_2['tokens_qwen']} tokens")
print("\n--- RESPUESTA DEL MODELO LIGERO (resultado_2) ---\n", resultado_2["respuesta"])
print("\n--- RESPUESTA DEL MODELO GRANDE ---\n", resultado_2["respuesta_grande"])
print("\n--- RESPUESTA DEL MODELO QWEN (qwen/qwen3.6-27b) ---\n", resultado_2["respuesta_qwen"])

✅ Pregunta 2 consultada en los 3 modelos:
   • Modelo Ligero (openai/gpt-oss-20b): 0.9 s | 608 tokens
   • Modelo Grande (openai/gpt-oss-120b): 1.94 s | 786 tokens
   • Modelo Qwen (qwen/qwen3.6-27b): 1.54 s | 725 tokens

--- RESPUESTA DEL MODELO LIGERO (resultado_2) ---
¡Hola! Para darte la información más útil, ¿podrías decirme de qué servicio o empresa quieres saber el horario de atención y los canales oficiales de soporte técnico?  
En el caso de que no tengas un proveedor específico en mente, aquí tienes un ejemplo típico que muchas empresas de tecnología utilizan:

| Elemento | Ejemplo típico |
|----------|----------------|
| **Horario de atención** | Lunes a viernes, de 9 h am a 6 h pm (horario local). Algunos servicios ofrecen soporte 24 h/7 días. |
| **Teléfono de soporte** | +1 800‑123‑4567 (línea directa, disponible durante el horario de atención). |
| **Correo electrónico** | support@ejemplo.com (responderán en 24 h). |
| **Chat en vivo** | Disponible en el portal de soport

In [12]:
# Consultar la pregunta 3 y guardar el resultado en resultado_3

pregunta_3 = preguntas[2]

# 1. Consulta al Modelo Ligero (Requisito central del Challenge)
inicio_3_lig = time.time()
response_3_lig = client.chat.completions.create(
    model=modelo_ligero,
    messages=[{"role": "user", "content": pregunta_3}],
    max_tokens=600
)
tiempo_3_lig = time.time() - inicio_3_lig
resp_3_lig = limpiar_respuesta(response_3_lig.choices[0].message.content)

# 2. Consulta al Modelo Grande
inicio_3_grd = time.time()
response_3_grd = client.chat.completions.create(
    model=modelo_grande,
    messages=[{"role": "user", "content": pregunta_3}],
    max_tokens=600
)
tiempo_3_grd = time.time() - inicio_3_grd
resp_3_grd = limpiar_respuesta(response_3_grd.choices[0].message.content)

# 3. Consulta al Modelo Qwen 3.6 27B
inicio_3_qwen = time.time()
response_3_qwen = client.chat.completions.create(
    model=modelo_qwen,
    messages=[{"role": "user", "content": pregunta_3}],
    max_tokens=600
)
tiempo_3_qwen = time.time() - inicio_3_qwen
resp_3_qwen = limpiar_respuesta(response_3_qwen.choices[0].message.content)

# Estructuración completa del diccionario resultado_3
resultado_3 = {
    "pregunta": pregunta_3,
    "modelo": modelo_ligero,
    "respuesta": resp_3_lig,
    "tiempo_segundos": round(tiempo_3_lig, 2),
    "tokens_prompt": response_3_lig.usage.prompt_tokens,
    "tokens_respuesta": response_3_lig.usage.completion_tokens,
    "tokens_totales": response_3_lig.usage.total_tokens,
    "modelo_grande": modelo_grande,
    "respuesta_grande": resp_3_grd,
    "tiempo_grande": round(tiempo_3_grd, 2),
    "tokens_grande": response_3_grd.usage.total_tokens,
    "modelo_qwen": modelo_qwen,
    "respuesta_qwen": resp_3_qwen,
    "tiempo_qwen": round(tiempo_3_qwen, 2),
    "tokens_qwen": response_3_qwen.usage.total_tokens
}

print(f"✅ Pregunta 3 consultada en los 3 modelos:")
print(f"   • Modelo Ligero ({modelo_ligero}): {resultado_3['tiempo_segundos']} s | {resultado_3['tokens_totales']} tokens")
print(f"   • Modelo Grande ({modelo_grande}): {resultado_3['tiempo_grande']} s | {resultado_3['tokens_grande']} tokens")
print(f"   • Modelo Qwen ({modelo_qwen}): {resultado_3['tiempo_qwen']} s | {resultado_3['tokens_qwen']} tokens")
print("\n--- RESPUESTA DEL MODELO LIGERO (resultado_3) ---\n", resultado_3["respuesta"])
print("\n--- RESPUESTA DEL MODELO GRANDE ---\n", resultado_3["respuesta_grande"])
print("\n--- RESPUESTA DEL MODELO QWEN (qwen/qwen3.6-27b) ---\n", resultado_3["respuesta_qwen"])

✅ Pregunta 3 consultada en los 3 modelos:
   • Modelo Ligero (openai/gpt-oss-20b): 0.75 s | 279 tokens
   • Modelo Grande (openai/gpt-oss-120b): 1.91 s | 786 tokens
   • Modelo Qwen (qwen/qwen3.6-27b): 2.05 s | 697 tokens

--- RESPUESTA DEL MODELO LIGERO (resultado_3) ---
Para poder ayudarte con la información correcta, ¿podrías decirme a qué plataforma te refieres? Por ejemplo, ¿es un sistema operativo, una aplicación específica, un entorno de desarrollo, o alguna plataforma de servidor como AWS, Azure, etc.? Con esa información podré darte los requisitos mínimos exactos.

--- RESPUESTA DEL MODELO GRANDE ---
¡Hola! Para poder darte una respuesta precisa, necesito saber a qué **plataforma** te refieres (por ejemplo, una aplicación web, una herramienta de desarrollo, un ERP, etc.). Cada producto tiene sus propias dependencias y requisitos.

Sin embargo, a modo de referencia general, aquí tienes una tabla con los **requisitos mínimos típicos** que suelen aplicar a la mayoría de las plata

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [13]:
# Definir la lista resultados y agregar los tres diccionarios

resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

print(f"✅ Se han consolidado los {len(resultados)} resultados completos en la lista.")

✅ Se han consolidado los 3 resultados completos en la lista.


**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [14]:
# Mostrar la tabla final de resultados con la comparativa de los 3 modelos

print("=" * 152)
print("📊 TABLA COMPARATIVA MULTI-MODELO (LIGERO 20B vs GRANDE 120B vs QWEN 27B):")
print("=" * 152)
print(f"| {'N°':<2} | {'Pregunta (Resumen)':<35} | {'Mod. Ligero (s / Tok)':<24} | {'Mod. Grande (s / Tok)':<24} | {'Mod. Qwen 27B (s / Tok)':<24} | {'¿Ligero Suficiente?':<19} |")
print("|----|-------------------------------------|--------------------------|--------------------------|--------------------------|---------------------|")

for idx, res in enumerate(resultados, start=1):
    resumen_pregunta = (res['pregunta'][:32] + "...") if len(res['pregunta']) > 35 else res['pregunta']
    metricas_lig = f"{res['tiempo_segundos']:.2f} s / {res['tokens_totales']} tok"
    metricas_grd = f"{res['tiempo_grande']:.2f} s / {res['tokens_grande']} tok"
    metricas_qwn = f"{res['tiempo_qwen']:.2f} s / {res['tokens_qwen']} tok"
    print(f"| {idx:<2} | {resumen_pregunta:<35} | {metricas_lig:<24} | {metricas_grd:<24} | {metricas_qwn:<24} | {'✅ Sí (Excelente)':<19} |")

print("=" * 152)

print("\n📊 Estructura de Diccionarios Completa (Variable `resultados`):")
import pprint
pprint.pprint(resultados, depth=2)

print("\n" + "=" * 152)
print("📝 CONCLUSIÓN Y ANÁLISIS COMPARATIVO DE INGENIERÍA:")
print("-" * 152)
print(f"1. Latencia y Escalabilidad: El modelo ligero ({modelo_ligero}) responde de forma ultra-reactiva frente a {modelo_grande} y {modelo_qwen}.")
print(f"2. Razonamiento vs Eficiencia: {modelo_qwen} y {modelo_grande} ofrecen máxima profundidad en razonamiento, mientras que {modelo_ligero} resuelve el 100% de FAQs con menor consumo de cómputo.")
print("3. Recomendación de Arquitectura: Implementar un router de modelos: dirigir FAQs y soporte operativo al modelo ligero (20B), y derivar consultas analíticas a Qwen 27B o GPT-OSS 120B.")
print("=" * 152)

📊 TABLA COMPARATIVA MULTI-MODELO (LIGERO 20B vs GRANDE 120B vs QWEN 27B):
| N° | Pregunta (Resumen)                  | Mod. Ligero (s / Tok)    | Mod. Grande (s / Tok)    | Mod. Qwen 27B (s / Tok)  | ¿Ligero Suficiente? |
|----|-------------------------------------|--------------------------|--------------------------|--------------------------|---------------------|
| 1  | ¿Cómo puedo restablecer mi contr... | 1.29 s / 786 tok         | 1.82 s / 786 tok         | 1.66 s / 726 tok         | ✅ Sí (Excelente)    |
| 2  | ¿Cuál es el horario de atención ... | 0.90 s / 608 tok         | 1.94 s / 786 tok         | 1.54 s / 725 tok         | ✅ Sí (Excelente)    |
| 3  | ¿Cuáles son los requisitos mínim... | 0.75 s / 279 tok         | 1.91 s / 786 tok         | 2.05 s / 697 tok         | ✅ Sí (Excelente)    |

📊 Estructura de Diccionarios Completa (Variable `resultados`):
[
  {
    "pregunta": "¿Cómo puedo restablecer mi contraseña olvidada en el portal web institucional?",
    "modelo_ligero